##### make_date()

- Used to Create a **DateType** column from **Year, Month and Day Columns.** 
- It is available in PySpark version **3.3.0 and later**. 

##### Syntax
     make_date(year: ColumnOrName, month: ColumnOrName, day: ColumnOrName)

|     Argument        |     Description              |
|---------------------|------------------------------|
| **Input Types**     | The `arguments` **year, month, and day** should be **integer** expressions (**either literal integers or columns of an integer type**). |
| **year**            | An `INTEGER` expression evaluating to a value from **1 to 9999**. |
| **month**           | An `INTEGER` expression evaluating to a value from **1 (January) to 12 (December)**. |
| **date**            | An `INTEGER` expression evaluating to a value from **1 to 31**. |
| **Return Type**     | The function returns a column of **DateType**. |
| **Error Handling**  | If any argument is **out of bounds** (e.g., **month 13 or day 32**), the result for that row is **NULL**. |
|                     | If the **input values form** an **invalid date** (e.g., `make_date(2019, 13, 1) or make_date(2019, 2, 30)`), the function will return **NULL**. |
| **Alternatives**    | For `older Spark versions`, a common `alternative` is to **concatenate the year, month, and day** columns into a **string** and then **cast to a date type** using functions like **concat_ws and cast("date")**. |

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, when, make_date

##### 1) Create a `Date Column` from Separate `Year, Month and Day Columns`

In [0]:
# Create a DataFrame with separate year, month, and day columns
data = [[2021, 10, 30], 
        [2021, 12, 3], 
        [2022, 1, 14], 
        [None, None, None],
        [2024, 2, 29],       # Valid date for a leap year 
        [2025, 2, 28],
        [2022, 5, 24], 
        [2023, 3, 21],
        [2023, 7, 18],
        [2023, 11, 4],
        [2023, 12, None]] 
  
# define column names
columns = ['year', 'month', 'day'] 

df_components = spark.createDataFrame(data, columns)
display(df_components)

year,month,day
2021,10,30
2021,12,3
2022,1,14
null,null,null
2024,2,29
2025,2,28
2022,5,24
2023,3,21
2023,7,18
2023,11,4


In [0]:
# Create a DateType column using the make_date() function
df_date = df_components.withColumn(
    "date",
    F.make_date(F.col("year"), F.col("month"), F.col("day"))
)

display(df_date)

year,month,day,date
2021,10,30,2021-10-30
2021,12,3,2021-12-03
2022,1,14,2022-01-14
null,null,null,null
2024,2,29,2024-02-29
2025,2,28,2025-02-28
2022,5,24,2022-05-24
2023,3,21,2023-03-21
2023,7,18,2023-07-18
2023,11,4,2023-11-04


##### 2) Using Literal Values

In [0]:
from pyspark.sql.functions import lit

spark.range(1).select("id",
    F.make_date(lit(2026), lit(1), lit(1)).alias("new_year"),
    F.make_date(lit(2026), lit(1), lit(None)).alias("new_year_null")
).display()

id,new_year,new_year_null
0,2026-01-01,null


##### 3) Handling Invalid Dates
- `Invalid dates` return `NULL`.

In [0]:
%sql
SELECT make_date(2013, 7, 15)
UNION ALL
SELECT make_date(2019, 7, NULL);

"make_date(2013,7,15)"
2013-07-15
null


In [0]:
%sql
SELECT make_date(2019, 13, 1) as invalid_date;

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-6743578884386974>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT make_date(2019, 13, 1) as invalid_date;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:192, in SqlMagic.sql(self, line, cell)
    186 except BaseException as e:
    187     self.driver_activit

In [0]:
%sql
SELECT make_date(2019, 2, 30) as invalid_date;

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-6743578884386975>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT make_date(2019, 2, 30) as invalid_date;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:192, in SqlMagic.sql(self, line, cell)
    186 except BaseException as e:
    187     self.driver_activit

In [0]:
data = [(2026, 2, 30)]

df_invalid = spark.createDataFrame(data, ["year", "month", "day"])
display(df_invalid)

df_invalid.select(F.make_date("year", "month", "day")).display()

year,month,day
2026,2,30


---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-6743578884386967>, line 6
      3 df_invalid = spark.createDataFrame(data, ["year", "month", "day"])
      4 display(df_invalid)
----> 6 df_invalid.select(F.make_date("year", "month", "day")).display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:74, in apply_dataframe_display_patch.<locals>.df_display(df, *args, **kwargs)
     70 def df_display(df, *args, **kwargs):
     71     """
     72     df.display() is an alias for display(df). Run help(display) for more information.
     73     """
---> 74     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif i

##### 4) Create a `Date Column` from Separate `Year, Month, and Day` Columns with `Error Handling`

In [0]:
# Create a DataFrame with separate year, month, and day columns
data = [(2022, 7, 25),
        (2023, 1, 1),
        (2022, 13, 31),
        (2023, 2, 28),
        (2023, 2, 29),
        (2024, 5, 19),
        (2025, 8, 29)]

df_invalid = spark.createDataFrame(data, ["year", "month", "day"])
display(df_invalid)

year,month,day
2022,7,25
2023,1,1
2022,13,31
2023,2,28
2023,2,29
2024,5,19
2025,8,29


In [0]:
# Create a DateType column using the make_date() function
df_invalid_final = df_invalid.withColumn("date", make_date(col("year"), col("month"), col("day")))

# Display the result
display(df_invalid_final)

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-6860096963566737>, line 5
      2 df_invalid_final = df_invalid.withColumn("date", make_date(col("year"), col("month"), col("day")))
      4 # Display the result
----> 5 display(df_invalid_final)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
    135     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:97, in Display.display_connect_table(self, df, **kwargs)
     94     self.cf_helper.display_streaming_dataframe(df, config, self.streaming_listener,
     95                                                **kwargs)
     96 else:
---> 

In [0]:
# spark.conf.set("spark.sql.ansi.enabled", True)

In [0]:
# spark.conf.set("spark.sql.ansi.enabled", True)
spark.conf.set("spark.sql.ansi.enabled", False)

# Create a DateType column using the make_date() function
df_invalid_final = df_invalid.withColumn("date", make_date(col("year"), col("month"), col("day")))

# Display the result
display(df_invalid_final)

year,month,day,date
2022,7,25,2022-07-25
2023,1,1,2023-01-01
2022,13,31,null
2023,2,28,2023-02-28
2023,2,29,null
2024,5,19,2024-05-19
2025,8,29,2025-08-29


##### 5) Create a `Date Column` from Separate `Year, Month, and Day` Columns with `Custom Error Handling`.

In [0]:
# Create a DataFrame with separate year, month, and day columns
data = [(2022, 7, 25),
        (2023, 1, 1),
        (2022, 13, 31),
        (2024, 5, 19),
        (2025, 8, 29)]

df_error = spark.createDataFrame(data, ["year", "month", "day"])
display(df_error)

year,month,day
2022,7,25
2023,1,1
2022,13,31
2024,5,19
2025,8,29


In [0]:
# Create a DateType column using the make_date() function with custom error handling
df_error_final = df_error \
.withColumn("date", when((col("month") >= 1) & (col("month") <= 12) & (col("day") >= 1) & (col("day") <= 31),
                          make_date(col("year"), col("month"), col("day"))).otherwise(lit(None)))

# Display the result
display(df_error_final)

year,month,day,date
2022,7,25,2022-07-25
2023,1,1,2023-01-01
2022,13,31,null
2024,5,19,2024-05-19
2025,8,29,2025-08-29


##### 6) Using in SQL

In [0]:
df_components.createOrReplaceTempView("date_table")

spark.sql("""
SELECT
    year,
    month,
    day,
    make_date(year, month, day) AS full_date
FROM date_table
""").display()

year,month,day,full_date
2021,10,30,2021-10-30
2021,12,3,2021-12-03
2022,1,14,2022-01-14
null,null,null,null
2024,2,29,2024-02-29
2025,2,28,2025-02-28
2022,5,24,2022-05-24
2023,3,21,2023-03-21
2023,7,18,2023-07-18
2023,11,4,2023-11-04


In [0]:
df_components.createOrReplaceTempView("date_table_01")

In [0]:
%sql
SELECT
    year,
    month,
    day,
    make_date(year, month, day) AS full_date
FROM date_table_01

year,month,day,full_date
2021,10,30,2021-10-30
2021,12,3,2021-12-03
2022,1,14,2022-01-14
null,null,null,null
2024,2,29,2024-02-29
2025,2,28,2025-02-28
2022,5,24,2022-05-24
2023,3,21,2023-03-21
2023,7,18,2023-07-18
2023,11,4,2023-11-04
